In [1]:
# Step 1. Set Google Colab Connection
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [1]:
# Step 2. Import Python Packages: Pandas, Numpy, Statistics, and Scipy
#   Optimization via Scipy

import pandas as pd
import numpy as np
import statistics as st
import scipy
from scipy import stats
import scipy.optimize as sci
from scipy.optimize import minimize
from scipy.optimize import fsolve

In [4]:
# Step 3. I/O Folders and Filenames

# I/O Folders
InputPath="/content/drive/MyDrive/KRG/"
OutputPath="/content/drive/MyDrive/KRG/"

# Inputfile: MI Datafile
MIDataFile="MIData_KRG.csv"

# OutputFile: MI Parmeters
MIResultsFile="MIParameters.csv"



In [6]:
# Step 4. Define Variables
#MIData = pd.read_csv(InputPath+MIDataFile)
MIData = pd.read_csv(MIDataFile)
Cost = MIData['Cost']
Size = MIData['Size']
Volatility = MIData['Volatility']
POV = MIData['POV']
Cost = MIData['Cost']
X = MIData[['Size','Volatility','POV']]
n,k = X.shape
  # n=number of observations
  # k=number of input variables

print(MIData)


             Date    Size  Volatility     POV        Cost
0       8/14/2020  0.0927    0.498829  0.0303   -0.357895
1       8/14/2020  0.0568    0.344376  0.0333  171.617139
2       8/14/2020  0.0534    0.193020  0.0415  -26.072693
3       8/14/2020  0.0580    0.471967  0.0478  260.844290
4       8/14/2020  0.1500    0.337130  0.0538  126.922787
...           ...     ...         ...     ...         ...
233130  6/30/2022  0.1768    0.525394  0.5871  197.113837
233131  6/30/2022  0.3853    0.322805  0.5872  124.846741
233132  6/30/2022  0.2236    0.261822  0.5887   95.185164
233133  6/30/2022  0.1311    0.132569  0.5944   62.545765
233134  6/30/2022  0.7262    0.457231  0.5979  154.009555

[233135 rows x 5 columns]


In [7]:
# Step 5. Definte MI Cost Optimization function
#   We are solving via non-linear least squares
#   Data is compiled from Investor/Broker data

def MI_NonLinearOLS(initial_guess, Size, POV, Volatility, Cost):
    a1, a2, a3, a4, b1 = initial_guess
    return np.sum(np.square((a1 * (Size ** a2) * (Volatility ** a3) * (b1 * (POV ** a4) + (1 - b1))) - Cost))


In [10]:
# Step 6. Run Non-Linear Regression using Non-Linear Least Squares
#         To estimate the MI Parameters

#Initial Guess for MI Parameter Values
a1 = 1000
a2 = 0.5
a3 = 0.5
a4 = 0.5
b1 = 0.95
initial_guess = [a1, a2, a3, a4, b1]

# Set Upper and Lower Bounds on Constraints
LB = [0, 0.0001, 0.0001, 0.0001, 0.00001]
UB = [2000, 2.0, 2.0, 2.0, 1]

# Write LB and UB in proper format
i = 0
MyBounds = []
for i in range(0,len(LB)):
    MyBounds.append((LB[i], UB[i]))
    i = i+1

#MyBounds2 = [(0,2000), (0.0001, 2.0), (0.0001, 2.0), (0.0001, 2.0), (0.0001, 1),]


# Run Non-Linear Regression using scipy minimize function
results = minimize(MI_NonLinearOLS, initial_guess, bounds = MyBounds, args = (Size, POV, Volatility, Cost))
print(results)

# MI Parameter Values
a1 = results.x[0]
a2 = results.x[1]
a3 = results.x[2]
a4 = results.x[3]
b1 = results.x[4]


  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 4945541207.806495
        x: [ 8.835e+02  3.540e-01  7.557e-01  8.260e-01  9.636e-01]
      nit: 25
      jac: [ 9.537e+01 -2.689e+04  1.268e+04  1.631e+04  2.413e+04]
     nfev: 174
     njev: 29
 hess_inv: <5x5 LbfgsInvHessProduct with dtype=float64>


In [11]:
print(a1, a2, a3, a4, b1)

883.5042520151342 0.3540152366892712 0.7556790195409708 0.8260439083519795 0.963582240900899


In [12]:
# Step 7. Calculate Model Error
#    Model error is not provided in our non-linear regression

# Calculate Estimated Cost
EstCost=(a1*Size**a2*Volatility**a3)*(b1*POV**a4+(1-b1))

# Actual Cost
ActualCost=Cost

# Calculate IStar Model Error
EstCost=(a1*Size**a2*Volatility**a3)*(b1*POV**a4+(1-b1))
MSE = np.mean((Cost - EstCost) ** 2)
SE=MSE**0.5

print(MSE, SE)

21213.20783154179 145.64754660323595


In [14]:
# Step 8. Save Results

DataResults = [[a1,a2,a3,a4,b1,MSE,SE]]
MIResults = pd.DataFrame(DataResults, columns=['a1','a2','a3','a4','b1','MSE','SE'])
#MIResults.to_csv(OutputPath+MIResultsFile, index = False, header=True)
MIResults.to_csv(MIResultsFile, index = False, header=True)
MIResults.head()

,a1,a2,a3,a4,b1,MSE,SE
0,883.504252,0.354015,0.755679,0.826044,0.963582,21213.207832,145.647547
